In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1) Загрузим данные (авто-детекция разделителя)
df = pd.read_csv('train.csv', sep=None, engine='python')

# 2) Обозначим целевой признак и признаки
target = 'Survived'
if target not in df.columns:
    raise SystemExit("Ошибка: в наборе данных отсутствует целевой столбец 'Survived'.")

X = df.drop(columns=[target])
y = df[target]

# 3) Простейшая инженерия признаков
if 'FamilySize' not in X.columns:
    X = X.copy()
    # FamilySize = количество членов семьи = SibSp + Parch + 1
    X['FamilySize'] = X['SibSp'] + X['Parch'] + 1

# 4) Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5) Бейзлайн: константное предсказание наиболее частого класса
most_freq = y_train.mode()[0]
baseline_pred = np.full(y_test.shape, most_freq)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Baseline accuracy (наиболее частый класс): {baseline_acc:.4f}")

# 6) Разделение признаков на категориальные и числовые
# Динамически выбираем признаки по типам данных
categorical_features = [c for c in X.columns if X[c].dtype == 'object']
numeric_features = [c for c in X.columns if X[c].dtype != 'object']

# 7) Предобработка:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 8) Модель
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
])

# 9) Обучение модели
clf.fit(X_train, y_train)

# 10) Оценка на тестовой выборке
y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_proba)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

print("Classification report (для порога 0.5):")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

Baseline accuracy (наиболее частый класс): 0.6145
Test accuracy: 0.8268
Test ROC-AUC: 0.8532
Classification report (для порога 0.5):
              precision    recall  f1-score   support

Not Survived       0.84      0.89      0.86       110
    Survived       0.81      0.72      0.76        69

    accuracy                           0.83       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.83      0.83      0.82       179



In [ ]:
from google.colab import drive
drive.mount('/content/drive')